In [1]:
%matplotlib ipympl

import pandas as pd
import numpy as np
import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)

In [2]:
def load_sms_references(file_path: str) -> tuple[jax.Array, jax.Array]:
    """Load reference linear accelerations and angular velocities."""
    df = pd.read_csv(file_path)

    acc_keys = [f"sesmt.md.merged_frame.xyz_acc[{i}] {{m/s^2}}" for i in range(3)]
    omega_keys = [f"sesmt.md.merged_frame.ang_vel[{i}] {{rad/s}}" for i in range(3)]
    gravity_keys = [
        f"sesmt.md.merged_frame.gravity[{i}] {{m/s^2}}" for i in range(3)
    ]

    acc_ref = jnp.array(df[acc_keys])
    omega_ref = jnp.array(df[omega_keys])
    gravity_ref = jnp.array(df[gravity_keys])

    # for some reason, data collection after a lot of nonsense data
    # we grab the data after we start recognizing nonzero (x) accelerations
    # note that using direct equality is desired here (and not jnp.isclose)
    offset = jnp.argmax(acc_ref[:, 0] != 0.0)

    # we need to cancel and then add back in the gravity vector
    acc_ref = acc_ref[offset:, :] - 2 * gravity_ref[offset:, :]
    omega_ref = omega_ref[offset:, :]
    
    return acc_ref, omega_ref

def load_specific_sms_references(file_path: str) -> tuple[jax.Array, jax.Array]:
    df = pd.read_csv(file_path)

    ks = df.keys()

    ts = np.array(df[ks[0]])
    diff = np.abs(np.diff(ts))
    avg_diff = np.mean(diff)
    std_diff = np.std(diff)
    if std_diff > 0.05:
        bad_indices = np.where(diff > avg_diff + std_diff)[0] + 1 # off by one
        start_index = bad_indices[-2] + 5 * 200
        end_index = bad_indices[-1] - 1
    else:
        start_index = 0
        end_index = ts.size - 1
    print(f"(start_index, end_index) = ({start_index}, {end_index})")

    # ts = ts[start_index: end_index + 1]
    df = df[start_index: end_index + 1]

    acc_ref = jnp.transpose(jnp.array([df[k] for k in ks[1:4]]))
    omega_ref = jnp.transpose(jnp.array([df[k] for k in ks[4:]]))
    return acc_ref, omega_ref

file_path = "/Users/jozbee/work/eng/comp/data/00_sms_drive.csv"
acc_ref, omega_ref = load_sms_references(file_path)

# file_path = "/Users/jozbee/work/eng/comp/data/specific-forces-lander-sim.csv"
# file_path = "/Users/jozbee/work/eng/comp/data/specific-forces-rover-sim.csv"
# file_path = "/Users/jozbee/work/eng/comp/data/specific-forces-standard-road-v2.csv"
# file_path = "/Users/jozbee/work/eng/comp/data/specific-forces-lander_motion_approach_auto.csv"
# file_path = "/Users/jozbee/work/eng/comp/data/specific-forces-lander_motion_redes_auto.csv"
# file_path = "/Users/jozbee/work/eng/comp/data/specific-forces-lander_motion_redes_manual.csv"
# acc_ref, omega_ref = load_specific_sms_references(file_path)

acc_ref = np.array(acc_ref)
omega_ref = np.array(omega_ref)

In [3]:
ts = 0.005 * np.arange(acc_ref.shape[0])
df = pd.DataFrame(
    data=np.column_stack([ts, acc_ref, omega_ref]),
    columns=["t", "accx", "accy", "accz", "omegax", "omegay", "omegaz"],
)

In [4]:
file_paths = file_path.split("/")
file_paths[-1] = f"{file_paths[-1].split(".")[0]}.hdf"
save_file_path = f"/{"/".join(file_paths[:-1])}/clean_{file_paths[-1]}"
df.to_hdf(save_file_path, key="data", mode="w")